# Preprocessing example

# 1. Import frameproc module and need other packages
frameproc is module for preprocessing step which is contain bias, dark, dark-sky flat field correction.

In [5]:
from pipeline.frameproc import Master, Process
from astropy.io import fits
from pipeline.utils import file_list, mkdir, save_fits
import ray
import sys

2026-04-21 16:30:03,455	INFO util.py:154 -- Missing packages: ['ipywidgets']. Run `pip install -U ipywidgets`, then restart the notebook server for rich notebook output.


# 2. set initial input parameter
Data files are must be located in folder which consisted "BIAS, DARK, LIGHT" folders.
**Caution** "BIAS, DARK, LIGHT" folder name must be capitalize!! If not, code cannot find the data files.

Set the object name which you observed. Now I use "NGC784" There is no gap between catalog name and number. If gap exist, code cannot find files.

extension type must be same your data file's extension. For example. your LIGHT(BIAS, or DARK) file's extension is '.fit', then, extension type input is '.fit'

In [7]:
path = '/volumes/ssd/u_test'
obj = 'NGC784'
ext_type = 1 #.fit is 1, .fits is 0. default is 0(.fits)

and then, define Master and process class.
I set the names 'master' and 'process'

In [8]:
master = Master(path,ext_type)
process = Process(path, obj, ext_type)

# 3. Bias and Dark subtraction
First, we remove the Bias and Dark noise. 

In [9]:
#bias, dark subtraction and amplifier glow masking
mkdir(path, 'process')
master.master_bias()
master.master_dark()
master.amp_mask()

mkdir(path,'db_subed')
process.db_sub(bias=master.bias, dark=master.dark)

# 4. Masking
Once Bias and Dark are removed from Light, mask the objects, like stars. galaxies, HII regions and etc, to get correct background data which is used to get master flat.

In this section, I used ray to reduce time using parallel processing. However, you can also use for loop.

In [11]:
#masking and master flat
mkdir(path, 'mask')
hdul_list = file_list(process.path + '/db_subed', ext_type=process.ext_type)

@ray.remote
def mask(hdul,i,pix,amp_r, amp_mask=True):
    hdu = fits.getdata(hdul)
    process.mask(hdu,i,pix,amp_r,amp_mask=amp_mask)

if filter == 'u':
    amp_mask=master.ampl_mask
else :
    amp_mask = True

ray.shutdown()
ray.init(num_cpus=6)
ray.get([mask.remote(hdul_list[i],i,pix=1.89,amp_r=300,amp_mask=amp_mask) for i in range(len(hdul_list))])
ray.shutdown()

2026-04-21 15:23:32,930	INFO worker.py:1951 -- Started a local Ray instance.


# 5. Flat-field correction
using mask and Light image, we can make master flat which is dark-sky flat to use flat-field correction

In [12]:
#flat-fielding
master.master_flat()
mkdir(path,'pp')
db_list = file_list(process.path+'/db_subed', ext_type=process.ext_type)
process.proc(db_list, master.flat)

# 6. Background sky subtraction
There is two sky modeling method, "polynomial" and "rbf". rbf only can be used by for loop. Because it is require so much computing process power. I recommand using "polynomial".

In [15]:
#sky subtraction
mkdir(path, 'sky_subed')
pp_list = file_list(process.path+'/pp', process.ext_type)
mask_list = file_list(process.path + '/mask', process.ext_type)
@ray.remote
def bkg_sub(pp_list, mask_list, i, order):
    data, hdr = process.sky_sub(pp_list,mask_list,i, order)
    n = format(i, '04')
    save_fits(process.path+'/sky_subed',process.obj+'_'+str(n),data=data,hdr=hdr,ext_type=process.ext_type)

ray.shutdown()
ray.init(num_cpus=6)
ray.get([bkg_sub.remote(pp_list,mask_list,i, order=2) for i in range(len(pp_list))])
ray.shutdown()

2026-04-21 15:26:14,972	INFO worker.py:1951 -- Started a local Ray instance.


# 7. Make astrometry command file
Finally, make astrometry command file, then preprocessing is done.

In [16]:
#astrometry.sh generate
process.astrometry(radius=1.5)
sys.exit()

SystemExit: 